In [4]:
from data_frame.exploration.schema_analyzer import SchemaAnalyzer
from pyspark.sql import types as T
from data_frame.spark_utils import get_spark

In [2]:
# Create DataFrame with complex schema
schema = T.StructType([
    T.StructField("id", T.IntegerType(), False),
    T.StructField("name", T.StringType(), True),
    T.StructField("address", T.StructType([
        T.StructField("street", T.StringType(), True),
        T.StructField("city", T.StringType(), True),
        T.StructField("zip", T.StringType(), True)
    ]), True),
    T.StructField("phones", T.ArrayType(T.StringType()), True),
    T.StructField("metadata", T.MapType(T.StringType(), T.StringType()), True)
])

In [3]:
data = [(1, "Alice", 
         T.Row(street="123 Main St", city="Boston", zip="02101"),
         ["555-1234", "555-5678"],
         {"source": "web", "verified": "true"})]

In [6]:
spark = get_spark(app_name="Schema Inspection")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/18 07:57:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
df = spark.createDataFrame(data, schema)

In [8]:
# Analyze schema
SchemaAnalyzer.print_schema_details(df)

SCHEMA DETAILS
root
 |-- id: integer (nullable = false)
 |-- name: string (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- street: string (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- zip: string (nullable = true)
 |-- phones: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- metadata: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



In [9]:
# Get column statistics
stats = SchemaAnalyzer.get_column_stats(df)
for col_name, col_stats in stats.items():
    print(f"\nColumn: {col_name}")
    for key, value in col_stats.items():
        print(f"  {key}: {value}")


Column: id
  type: int
  nullable: False
  metadata: {}

Column: name
  type: string
  nullable: True
  metadata: {}

Column: address
  type: struct<street:string,city:string,zip:string>
  nullable: True
  metadata: {}

Column: phones
  type: array<string>
  nullable: True
  metadata: {}

Column: metadata
  type: map<string,string>
  nullable: True
  metadata: {}


In [10]:
# Sample data
SchemaAnalyzer.sample_data(df)

SAMPLE DATA (First 5 rows)


+---+-----+----------------------------+--------------------+---------------------------------+
|id |name |address                     |phones              |metadata                         |
+---+-----+----------------------------+--------------------+---------------------------------+
|1  |Alice|{123 Main St, Boston, 02101}|[555-1234, 555-5678]|{verified -> true, source -> web}|
+---+-----+----------------------------+--------------------+---------------------------------+

